<div dir="rtl">

# חיפוש סמנטי ומערכת שאלה-תשובה בעברית: אימון מודל Embedding על HeQ

המטרה: לבנות מנוע חיפוש שמקבל **שאלה בעברית** ומוצא את **הפסקה שמכילה את התשובה**, ואז לשפר אותו באמצעות אימון.

מה המחברת עושה, מלמעלה למטה:
1. טוענת את מאגר **HeQ** (כ-30 אלף שאלות על פסקאות מוויקיפדיה העברית ומ-Geektime).
2. מודדת מודלים קיימים: חיפוש מילים קלאסי (BM25) ושלושה מודלי Embedding רב-לשוניים.
3. מכינה **שליליים קשים**: פסקאות שנראות קשורות לשאלה אבל לא מכילות את התשובה.
4. **מאמנת** (fine-tuning) את `multilingual-e5-base` על שאלות האימון.
5. משווה את כל המודלים על חלק הבדיקה, שהמודל לא ראה באימון, ומציגה דוגמאות.
6. מעלה את המודל ואת הנתונים ל-Hugging Face, בשביל אפליקציית הדמו.

**הרצה:** `Runtime → Change runtime type → T4 GPU`, ואז `Runtime → Run all`. כל ההרצה לוקחת בערך שעה, ורובה זמן האימון.

</div>

<div dir="rtl">

## 1. התקנה והגדרות

כל הפרמטרים החשובים נמצאים בתא ההגדרות. אפשר לשנות בו את מודל הבסיס, מספר ה-epochs או גודל ה-batch.

</div>

In [ ]:
!pip install -q -U sentence-transformers rank_bm25

In [ ]:
import json, random, re, os, time
import numpy as np, pandas as pd, torch
from collections import defaultdict
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "")

# ---- Settings ----
BASE_MODEL = "intfloat/multilingual-e5-base"   # the model we fine-tune
FINETUNED_NAME = "hebrew-qa-retriever-e5-base"  # name of the trained model on the Hub
DATASET_NAME = "hebrew-qa-retrieval-data"       # name of the data repo on the Hub
EPOCHS = 2
BATCH_SIZE = 128          # questions per step; larger batch = more in-batch negatives
MINI_BATCH = 32           # what actually fits in GPU memory at once (gradient caching)
LR = 2e-5
MAX_LEN = 384             # tokens per paragraph
HARD_NEG_POOL = 10        # pick each hard negative from the top-N candidates

<div dir="rtl">

## 2. טעינת המאגר HeQ

HeQ בנוי כמו SQuAD: לכל פסקה יש כמה שאלות. המאגר כבר מחולק ל-train / val / test **לפי ערך**, כך שאין פסקה או ערך שמופיעים ביותר מחלק אחד. זה חשוב: המודל נבדק על נושאים שלא ראה.

לחיפוש משתמשים רק בשאלות **שיש להן תשובה בפסקה**. ב-HeQ יש גם שאלות בלי תשובה, ולהן אין "פסקה נכונה" למצוא.

**מאגר החיפוש** (corpus) הוא כל הפסקאות מכל החלקים, כ-4,900 פסקאות. כך כל שאלת בדיקה צריכה למצוא את הפסקה הנכונה מתוך כל המאגר, כמו במנוע חיפוש אמיתי.

</div>

In [ ]:
!git clone -q --depth 1 https://github.com/NNLP-IL/Hebrew-Question-Answering-Dataset heq 2>/dev/null || echo "already cloned"

corpus, para_id = [], {}            # corpus: list of dicts; para_id: text -> index
queries = {"train": [], "val": [], "test": []}

for split, fname in [("train", "train v1.1.json"), ("val", "val v1.1.json"), ("test", "test v1.1.json")]:
    data = json.load(open(f"heq/data/data v1.1/{fname}", encoding="utf-8"))["data"]
    for article in data:
        for p in article["paragraphs"]:
            text = p["context"].strip()
            if text not in para_id:
                para_id[text] = len(corpus)
                corpus.append({"id": len(corpus), "title": article["title"],
                               "source": article["source"], "split": split, "text": text})
            pid = para_id[text]
            for q in p["qas"]:
                if q["is_impossible"]:
                    continue
                queries[split].append({"qid": q["id"], "question": q["question"].strip(),
                                       "pid": pid, "source": article["source"]})

corpus_df = pd.DataFrame(corpus)
print(f"Paragraphs in corpus: {len(corpus):,}")
for s in queries:
    print(f"{s:5s}: {len(queries[s]):,} answerable questions")
print()
ex = queries["test"][0]
print("Example question:", ex["question"])
print("Its paragraph:   ", corpus[ex["pid"]]["text"][:200], "...")

<div dir="rtl">

## 3. פונקציית מדידה

לכל שאלה מדרגים את כל הפסקאות במאגר ובודקים **באיזה מקום** נמצאת הפסקה הנכונה. מזה מחשבים:

- **R@1, R@5, R@10**: אחוז השאלות שבהן הפסקה הנכונה נמצאת ב-1, 5 או 10 התוצאות הראשונות. R@5 הוא המדד הכי חשוב ל-RAG, כי בדרך כלל מעבירים למודל השפה 3 עד 5 פסקאות.
- **MRR@10**: ממוצע של 1 חלקי המיקום. מקום ראשון = 1, מקום שני = 0.5, וכן הלאה. מתגמל דירוג גבוה.
- **NDCG@10**: מדד מקובל בתחרויות חיפוש. כשיש פסקה נכונה אחת, הוא דומה ל-MRR.

את כל המדדים מחשבים גם בנפרד לוויקיפדיה ול-Geektime, כי סגנון הטקסט שונה.

</div>

In [ ]:
def metrics_from_ranks(ranks):
    r = np.asarray(ranks)
    return {"R@1": np.mean(r <= 1) * 100, "R@5": np.mean(r <= 5) * 100, "R@10": np.mean(r <= 10) * 100,
            "MRR@10": np.mean(np.where(r <= 10, 1 / r, 0)) * 100,
            "NDCG@10": np.mean(np.where(r <= 10, 1 / np.log2(r + 1), 0)) * 100}

def ranks_from_scores(scores, pos):
    # scores: [n_queries, n_corpus] tensor; rank of the correct paragraph (1 = top)
    pos = torch.as_tensor(pos, device=scores.device)
    pos_scores = scores[torch.arange(len(pos), device=scores.device), pos]
    return ((scores > pos_scores[:, None]).sum(1) + 1).cpu().numpy()

all_results, all_ranks = [], {}

def report(name, ranks, qs):
    all_ranks[name] = ranks
    row = {"Model": name, **metrics_from_ranks(ranks)}
    for src in ["Wikipedia", "Geektime"]:
        mask = np.array([q["source"] == src for q in qs])
        row[f"R@5 {src}"] = metrics_from_ranks(ranks[mask])["R@5"]
    all_results[:] = [r for r in all_results if r["Model"] != name] + [row]
    display(pd.DataFrame(all_results).set_index("Model").round(1))

<div dir="rtl">

## 4. קו בסיס: מודלים קיימים, בלי אימון

**BM25** הוא חיפוש מילים קלאסי, כמו במנועי חיפוש ישנים. הוא חזק כשהשאלה משתמשת באותן מילים כמו הפסקה, אבל בעברית יש לו בעיה: "בבית", "הבית" ו"לבית" הן בשבילו שלוש מילים שונות.

**מודלי Embedding** הופכים כל טקסט לווקטור, וטקסטים עם משמעות דומה מקבלים וקטורים קרובים. בודקים שלושה:
- `multilingual-e5-base` (278M פרמטרים): המודל שנאמן בהמשך.
- `multilingual-e5-large` (560M): גרסה גדולה פי שניים, כדי לבדוק אם המודל הקטן אחרי אימון עוקף מודל גדול בלי אימון.
- `Qwen3-Embedding-0.6B`: מודל חדש וחזק במיוחד.

מודלי e5 דורשים לכתוב `query:` לפני שאלה ו-`passage:` לפני פסקה. ל-Qwen3 יש הוראה מובנית לשאלות.

</div>

In [ ]:
from rank_bm25 import BM25Okapi

tokenize = lambda t: re.findall(r"\w+", t.lower())
bm25 = BM25Okapi([tokenize(p["text"]) for p in corpus])
test_q = queries["test"]
bm25_scores = torch.tensor(np.array([bm25.get_scores(tokenize(q["question"])) for q in tqdm(test_q, desc="BM25")]))
report("BM25 (keyword search)", ranks_from_scores(bm25_scores, [q["pid"] for q in test_q]), test_q)

In [ ]:
MODELS = {
    "multilingual-e5-base":  dict(path="intfloat/multilingual-e5-base",  q=dict(prompt="query: "),   d=dict(prompt="passage: ")),
    "multilingual-e5-large": dict(path="intfloat/multilingual-e5-large", q=dict(prompt="query: "),   d=dict(prompt="passage: ")),
    "Qwen3-Embedding-0.6B":  dict(path="Qwen/Qwen3-Embedding-0.6B",      q=dict(prompt_name="query"), d=dict()),
}
corpus_texts = [p["text"] for p in corpus]

def encode(model, texts, kwargs, bs=64):
    return model.encode(texts, batch_size=bs, convert_to_tensor=True, normalize_embeddings=True,
                        show_progress_bar=True, **kwargs)

def evaluate_model(name, model, cfg, qs=None):
    qs = qs or queries["test"]
    model.max_seq_length = 512
    C = encode(model, corpus_texts, cfg["d"])
    Q = encode(model, [q["question"] for q in qs], cfg["q"])
    report(name, ranks_from_scores(Q @ C.T, [q["pid"] for q in qs]), qs)
    return C

corpus_emb = {}
for name, cfg in MODELS.items():
    model = SentenceTransformer(cfg["path"], device=device)
    if device == "cuda": model.half()   # faster inference; training below uses full precision
    corpus_emb[name] = evaluate_model(name, model, cfg).float().cpu().numpy()
    del model; torch.cuda.empty_cache()

<div dir="rtl">

## 5. הכנת נתוני האימון ושליליים קשים

המודל לומד מזוגות של **שאלה ← הפסקה הנכונה**. בכל צעד אימון הוא רואה 128 שאלות, וכל פסקה של שאלה אחרת משמשת "שלילי" לשאלה הנוכחית. אבל שלילי כזה הוא בדרך כלל קל: פסקה על כדורגל מול שאלה על ביולוגיה.

**שלילי קשה** הוא פסקה שנראית קשורה אבל לא מכילה את התשובה. אותם מוצאים כך:
1. מחפשים כל שאלת אימון עם מודל הבסיס, ולוקחים את הפסקאות הכי דומות.
2. **מסננים "שליליים שקריים"**: פסקאות מאותו ערך, ופסקאות שקיבלו ציון גבוה מ-95% מהציון של הפסקה הנכונה. הן עלולות להכיל את התשובה, ולא נרצה ללמד את המודל שהן שגויות.
3. בוחרים באקראי אחת מ-10 המועמדות הקשות שנשארו.

החיפוש נעשה רק בתוך פסקאות האימון, כדי שפסקאות הבדיקה לא ייכנסו לאימון בשום צורה.

</div>

In [ ]:
cfg = MODELS["multilingual-e5-base"]
model = SentenceTransformer(BASE_MODEL, device=device)
if device == "cuda": model.half()
train_pids = [p["id"] for p in corpus if p["split"] == "train"]
C_train = torch.tensor(corpus_emb["multilingual-e5-base"][train_pids], device=device)
Q_train = encode(model, [q["question"] for q in queries["train"]], cfg["q"]).float()
del model; torch.cuda.empty_cache()

train_index = {pid: j for j, pid in enumerate(train_pids)}
scores = Q_train @ C_train.T
top_scores, top_idx = scores.topk(50, dim=1)
train_rows, no_neg = [], 0
for i, q in enumerate(tqdm(queries["train"], desc="Mining hard negatives")):
    pos_score = float(scores[i, train_index[q["pid"]]])
    pos_title = corpus[q["pid"]]["title"]
    cands = []
    for s, j in zip(top_scores[i].tolist(), top_idx[i].tolist()):
        pid = train_pids[j]
        # skip the correct paragraph, paragraphs from the same article, and anything scoring
        # above 95% of the correct paragraph (likely to also contain the answer)
        if pid != q["pid"] and corpus[pid]["title"] != pos_title and s < 0.95 * pos_score:
            cands.append(pid)
    if not cands:
        no_neg += 1
        continue
    train_rows.append({"anchor": "query: " + q["question"],
                       "positive": "passage: " + corpus[q["pid"]]["text"],
                       "negative": "passage: " + corpus[random.choice(cands[:HARD_NEG_POOL])]["text"]})
print(f"Training triplets: {len(train_rows):,}  (skipped {no_neg} questions with no safe negative)")
print("\nExample:\n", json.dumps(train_rows[0], ensure_ascii=False, indent=1)[:900])

<div dir="rtl">

## 6. אימון

- **Loss:** `CachedMultipleNegativesRankingLoss`. לכל שאלה, המודל צריך לתת לפסקה הנכונה ציון גבוה יותר מכל שאר הפסקאות ב-batch, כולל השליליים הקשים. הגרסה ה"Cached" מאפשרת batch גדול (128) גם על GPU קטן, כי היא מחשבת אותו בחלקים של 32.
- **בלי כפילויות ב-batch:** ב-HeQ יש בממוצע כמה שאלות לכל פסקה. אם שתי שאלות על אותה פסקה ייכנסו לאותו batch, הפסקה הנכונה של אחת תיחשב "שלילית" לשנייה. `NO_DUPLICATES` מונע את זה.
- **מעקב:** כל 50 צעדים המודל נמדד על חלק ה-validation, בלי לגעת בחלק הבדיקה.

</div>

In [ ]:
from datasets import Dataset
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.losses import CachedMultipleNegativesRankingLoss
from sentence_transformers.training_args import BatchSamplers
from sentence_transformers.evaluation import InformationRetrievalEvaluator

model = SentenceTransformer(BASE_MODEL, device=device)
model.max_seq_length = MAX_LEN

# Validation: val questions searched against all non-test paragraphs
val_corpus = {str(p["id"]): "passage: " + p["text"] for p in corpus if p["split"] != "test"}
val_evaluator = InformationRetrievalEvaluator(
    queries={q["qid"]: "query: " + q["question"] for q in queries["val"]},
    corpus=val_corpus,
    relevant_docs={q["qid"]: {str(q["pid"])} for q in queries["val"]},
    name="heq-val", mrr_at_k=[10], ndcg_at_k=[10], accuracy_at_k=[1, 5, 10], precision_recall_at_k=[10],
    show_progress_bar=False)
print("Before training:", {k: round(v, 4) for k, v in val_evaluator(model).items() if "ndcg" in k or "accuracy@5" in k})

args = SentenceTransformerTrainingArguments(
    output_dir="checkpoints", num_train_epochs=EPOCHS, per_device_train_batch_size=BATCH_SIZE,
    learning_rate=LR, warmup_ratio=0.1, fp16=(device == "cuda"),
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    eval_strategy="steps", eval_steps=50, save_strategy="no", logging_steps=10,
    report_to="none", seed=SEED)

trainer = SentenceTransformerTrainer(
    model=model, args=args,
    train_dataset=Dataset.from_list(train_rows).shuffle(seed=SEED),
    loss=CachedMultipleNegativesRankingLoss(model, mini_batch_size=MINI_BATCH),
    evaluator=val_evaluator)
start = time.time()
trainer.train()
print(f"Training took {(time.time() - start) / 60:.1f} minutes")
model.save("finetuned-model")

<div dir="rtl">

## 7. התוצאות

עכשיו מודדים את המודל המאומן על חלק הבדיקה, באותה דרך בדיוק כמו את המודלים הקודמים.

</div>

In [ ]:
FT = "Fine-tuned e5-base (ours)"
corpus_emb[FT] = evaluate_model(FT, model, MODELS["multilingual-e5-base"]).float().cpu().numpy()
results = pd.DataFrame(all_results).set_index("Model").round(1)
results.to_csv("results.csv")
results

In [ ]:
import matplotlib.pyplot as plt
ax = results[["R@1", "R@5", "R@10"]].plot.barh(figsize=(9, 4.5), width=0.8)
ax.set_xlabel("% of test questions where the correct paragraph was found")
ax.set_title("HeQ test set: retrieval accuracy")
ax.invert_yaxis(); ax.set_xlim(0, 100); ax.legend(loc="lower right")
for c in ax.containers: ax.bar_label(c, fmt="%.0f", fontsize=8)
plt.tight_layout(); plt.savefig("results.png", dpi=150); plt.show()

<div dir="rtl">

### דוגמאות: איפה האימון עזר?

שאלות שבהן מודל הבסיס לא דירג את הפסקה הנכונה ראשונה, והמודל המאומן כן. ולצורך יושר, גם כמה דוגמאות הפוכות.

</div>

In [ ]:
base_r, ft_r = all_ranks["multilingual-e5-base"], all_ranks[FT]
def show(idx, title):
    print(f"=== {title} ===")
    for i in idx[:5]:
        q = queries["test"][i]
        print(f"Q: {q['question']}\n   base rank: {base_r[i]}  ->  fine-tuned rank: {ft_r[i]}")
        print(f"   paragraph: {corpus[q['pid']]['text'][:150]}...\n")
improved = np.where((ft_r == 1) & (base_r > 1))[0]
worse = np.where((ft_r > 1) & (base_r == 1))[0]
print(f"Improved to rank 1: {len(improved)} questions | Dropped from rank 1: {len(worse)} questions\n")
show(improved, "Improved"); show(worse, "Got worse")

<div dir="rtl">

## 8. העלאה ל-Hugging Face

מעלים שני דברים:
1. **המודל המאומן**, כ-Model עם כרטיס שמסביר איך אומן ומה התוצאות.
2. **Dataset** עם הפסקאות, הווקטורים של שני המודלים, טבלת התוצאות ושאלות לדוגמה. אפליקציית הדמו טוענת משם הכל, כך שהיא לא צריכה לחשב וקטורים בעצמה.

צריך טוקן עם הרשאת **Write** מ-https://huggingface.co/settings/tokens.

</div>

In [ ]:
from huggingface_hub import login, HfApi
login()
api = HfApi()
user = api.whoami()["name"]
model_repo, data_repo = f"{user}/{FINETUNED_NAME}", f"{user}/{DATASET_NAME}"

# ---- Model card ----
table = results.reset_index().to_markdown(index=False)
card = f'''---
language: [he]
license: mit
library_name: sentence-transformers
base_model: {BASE_MODEL}
datasets: [HeQ]
tags: [sentence-transformers, retrieval, hebrew, rag]
pipeline_tag: sentence-similarity
---

# Hebrew QA Retriever (fine-tuned multilingual-e5-base)

Retrieves the Hebrew paragraph that answers a Hebrew question. Fine-tuned from `{BASE_MODEL}` on the
answerable questions of [HeQ](https://github.com/NNLP-IL/Hebrew-Question-Answering-Dataset), with mined hard negatives
and `CachedMultipleNegativesRankingLoss` (batch {BATCH_SIZE}, {EPOCHS} epochs, lr {LR}).

## Usage

```python
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("{model_repo}")
q = model.encode("query: מי ניסח את מגילת העצמאות?", normalize_embeddings=True)
p = model.encode(["passage: ..."], normalize_embeddings=True)
```
Use the `query: ` and `passage: ` prefixes, as with the base model.

## Results (HeQ test set, {len(queries["test"]):,} questions, search over all {len(corpus):,} paragraphs)

{table}

Test articles are disjoint from training articles. HeQ is licensed CC BY 4.0 (Webiks / MAFAT). Limitation: HeQ questions were written while looking at
the paragraph, so they share more wording with it than real user queries would.
'''
model.push_to_hub(model_repo, exist_ok=True)
api.upload_file(path_or_fileobj=card.encode(), path_in_repo="README.md", repo_id=model_repo)

# ---- Data for the demo app ----
os.makedirs("hub_data", exist_ok=True)
corpus_df.to_parquet("hub_data/corpus.parquet")
np.save("hub_data/emb_base.npy", corpus_emb["multilingual-e5-base"].astype(np.float16))
np.save("hub_data/emb_finetuned.npy", corpus_emb[FT].astype(np.float16))
results.to_csv("hub_data/results.csv")
examples = [{"question": queries["test"][i]["question"], "pid": int(queries["test"][i]["pid"])}
            for i in list(improved[:6]) + random.sample(range(len(queries["test"])), 6)]
json.dump(examples, open("hub_data/examples.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
api.create_repo(data_repo, repo_type="dataset", exist_ok=True)
api.upload_folder(folder_path="hub_data", repo_id=data_repo, repo_type="dataset")
print("Model:  ", f"https://huggingface.co/{model_repo}")
print("Dataset:", f"https://huggingface.co/datasets/{data_repo}")

<div dir="rtl">

## סיכום

- טבלת התוצאות והגרף נשמרו ב-`results.csv` וב-`results.png`.
- המודל והנתונים נמצאים ב-Hugging Face, בקישורים שהודפסו למעלה.
- **הסתייגות חשובה:** השאלות ב-HeQ נכתבו כשהכותבים ראו את הפסקה, ולכן הן חולקות איתה יותר מילים משאילתות אמיתיות של משתמשים. זה מיטיב במיוחד עם BM25, ולכן כדאי לבדוק את המודל גם על מאגר חיצוני.

</div>